In [10]:
import time
import json
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import requests
from urllib.parse import urlparse
from cerebras.cloud.sdk import Cerebras

# Set year to filter
target_year = "2023"

# Setup Cerebras client
client = Cerebras(
    api_key=os.environ.get("CEREBRAS_API_KEY")
)

# Setup Selenium
driver = webdriver.Chrome()
driver.get("https://media.kkr.com/")
wait = WebDriverWait(driver, 10)

def analyze_article_with_llm(article_text):
    """Analyze article text using Cerebras LLM and return structured data"""
    if not article_text or article_text == "To Review PDF":
        return {
            "price": None,
            "revenue": None,
            "location": None,
            "seller": None,
            "buyer": None,
            "asset_status": None,
            "transaction_status": None,
            "asset_name": None
        }
    
    try:
        system_prompt = """You are an expert financial analyst. Extract transaction information from press releases and return it in the exact JSON format requested. If information is not available, use null for that field. For numerical values, return only the number without currency symbols or units.
        It is very important that you return values, even if you are not 100 per sure, you will need to do assumptions based on the context provided."""
        
        user_prompt = f"""Get from the following article the information with the following JsonSchema: 
{{
  "type": "object",
  "properties": {{
    "price": {{ "type": "number" }},
    "revenue": {{ "type": "number" }},
    "location": {{ "type": "string" }},
    "seller": {{ "type": "string" }},
    "buyer": {{ "type": "string" }},
    "asset_status": {{ "type": "string" }},
    "transaction_status": {{ "type": "string" }},
    "asset_name": {{ "type": "string" }}
  }},
  "required": [
    "price",
    "revenue",
    "location",
    "seller",
    "buyer",
    "asset_status",
    "transaction_status",
    "asset_name"
  ]
}}

Article text:
{article_text}

Return only valid JSON without any additional text or formatting."""

        # Create streaming completion with your API config
        stream = client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user", 
                    "content": user_prompt
                }
            ],
            model="llama3.1-8b",
            stream=True,
            max_completion_tokens=2048,
            temperature=0.2,
            top_p=1
        )
        
        # Collect streaming response
        llm_response = ""
        print("🤖 Streaming LLM response...", end="")
        for chunk in stream:
            content = chunk.choices[0].delta.content or ""
            llm_response += content
            if content:
                print(".", end="", flush=True)
        
        print(" Done!")
        llm_response = llm_response.strip()
        
        # Try to parse the JSON response
        try:
            analysis_data = json.loads(llm_response)
            print(f"✅ LLM analysis completed successfully")
            return analysis_data
        except json.JSONDecodeError as e:
            print(f"❌ Error parsing LLM JSON response: {e}")
            print(f"Raw response: {llm_response}")
            return {
                "price": None,
                "revenue": None,
                "location": None,
                "seller": None,
                "buyer": None,
                "asset_status": None,
                "transaction_status": None,
                "asset_name": None
            }
            
    except Exception as e:
        print(f"❌ Error calling Cerebras API: {e}")
        return {
            "price": None,
            "revenue": None,
            "location": None,
            "seller": None,
            "buyer": None,
            "asset_status": None,
            "transaction_status": None,
            "asset_name": None
        }

# === STEP 1: Apply year filter ===
try:
    print(f"Applying year filter for {target_year}...")
    
    # Wait for the page to load and find the year dropdown
    year_dropdown = wait.until(EC.element_to_be_clickable(
        (By.CSS_SELECTOR, "#mediaNewsLabelGroup .selectric")
    ))
    
    # Click to open the dropdown
    driver.execute_script("arguments[0].click();", year_dropdown)
    time.sleep(4)  # Wait for dropdown to open
    
    # Find and click the target year option
    year_option = wait.until(EC.element_to_be_clickable(
        (By.XPATH, f"//li[contains(text(), '{target_year}')]")
    ))
    driver.execute_script("arguments[0].click();", year_option)
    
    print(f"✅ Successfully selected year {target_year}")
    time.sleep(1)  # Wait for content to reload
    
except (TimeoutException, NoSuchElementException) as e:
    print(f"❌ Error applying year filter: {e}")
    print("Continuing without year filter...")

# === STEP 2: Wait for first press release to load ===
wait.until(EC.presence_of_element_located((By.CLASS_NAME, "press__section")))

# === STEP 3: Load more (limited for testing) ===
click_count = 0
max_clicks = 1

while click_count < max_clicks:
    try:
        load_more = wait.until(EC.presence_of_element_located(
            (By.CSS_SELECTOR, "button.loadmore_button_kkr.active")
        ))
        driver.execute_script("arguments[0].click();", load_more)
        click_count += 1
        print(f"Clicked 'Load more' ({click_count}/{max_clicks})...")
        time.sleep(2)
    except:
        print("No more 'Load more' button or error.")
        break

def is_pdf_link(url):
    """Check if the URL points to a PDF file"""
    try:
        # Check file extension in URL
        parsed_url = urlparse(url)
        if parsed_url.path.lower().endswith('.pdf'):
            return True
        
        # Make a HEAD request to check content type
        response = requests.head(url, timeout=5, allow_redirects=True)
        content_type = response.headers.get('content-type', '').lower()
        
        # Check if content type indicates PDF
        if 'application/pdf' in content_type:
            return True
            
        # Check content-disposition header for PDF filename
        content_disposition = response.headers.get('content-disposition', '').lower()
        if '.pdf' in content_disposition:
            return True
            
        return False
    except Exception as e:
        print(f"Error checking if URL is PDF: {e}")
        return False

# === STEP 4: Scrape press releases ===
press_releases = driver.find_elements(By.CLASS_NAME, "press__section")
data = []

for i, release in enumerate(press_releases, 1):
    try:
        print(f"\n📰 Processing article {i}/{len(press_releases)}...")
        
        a_tag = release.find_element(By.CLASS_NAME, "press__link")
        date = a_tag.find_element(By.CLASS_NAME, "press--date").text.strip()
        full_text = a_tag.text.strip()
        title = full_text.replace(date, '').replace('|', '').strip()
        href = a_tag.get_attribute("href")
        full_link = href if href.startswith("http") else f"https://media.kkr.com{href}"

        print(f"📄 Title: {title}")
        
        # Check if the link points to a PDF file
        if is_pdf_link(full_link):
            print(f"📄 PDF detected for: {title}")
            article_text = "To Review PDF"
        else:
            # Open article page in new tab
            driver.execute_script("window.open(arguments[0]);", full_link)
            time.sleep(1)
            driver.switch_to.window(driver.window_handles[1])

            try:
                # Wait for page to load and check if it's actually an article page
                wait.until(EC.presence_of_element_located((By.CLASS_NAME, "news_details")))
                paragraphs = driver.find_elements(By.CSS_SELECTOR, ".news_details p")
                article_text = "\n".join(p.text.strip() for p in paragraphs if p.text.strip())
                
                # If no article text found, it might be a PDF or other file type
                if not article_text:
                    print(f"📄 No article text found, likely a file: {title}")
                    article_text = "To Review PDF"
                    
            except Exception as e:
                print(f"Error getting article text for {title}: {e}")
                # Check if the current URL indicates a PDF download
                current_url = driver.current_url
                if is_pdf_link(current_url) or current_url != full_link:
                    print(f"📄 Redirected to file download: {title}")
                    article_text = "To Review PDF"
                else:
                    article_text = ""

            driver.close()
            driver.switch_to.window(driver.window_handles[0])

        # Analyze article with LLM
        print("🤖 Analyzing with LLM...")
        llm_analysis = analyze_article_with_llm(article_text)
        
        # Combine all data
        article_data = {
            "date": date,
            "title": title,
            "link": full_link,
            "text": article_text,
            "analysis": llm_analysis
        }
        
        data.append(article_data)
        print(f"✅ Article {i} processed successfully")

    except Exception as e:
        print(f"❌ Error processing article {i}: {e}")
        # Ensure we're back on the main window if something went wrong
        if len(driver.window_handles) > 1:
            driver.close()
            driver.switch_to.window(driver.window_handles[0])

# === STEP 5: Save to JSON ===
filename = f"kkr_press_releases_{target_year}_with_analysis.json"
with open(filename, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"\n✅ Saved {len(data)} articles to {filename}")

# Print summary
pdf_count = sum(1 for item in data if item["text"] == "To Review PDF")
analyzed_count = len(data) - pdf_count

if pdf_count > 0:
    print(f"📄 Found {pdf_count} PDF files that need manual review")
if analyzed_count > 0:
    print(f"🤖 Analyzed {analyzed_count} articles with LLM")

# Show sample analysis results
transactions_found = sum(1 for item in data if item["analysis"]["asset_name"] is not None)
if transactions_found > 0:
    print(f"💼 Found {transactions_found} potential transactions")

# === Cleanup ===
driver.quit()

Applying year filter for 2023...
✅ Successfully selected year 2023
Clicked 'Load more' (1/1)...

📰 Processing article 1/30...
📄 Title: KKR Announces Intra-Quarter Monetization Activity Update for the Fourth Quarter
🤖 Analyzing with LLM...
🤖 Streaming LLM response......................................................... Done!
✅ LLM analysis completed successfully
✅ Article 1 processed successfully

📰 Processing article 2/30...
📄 Title: ETCHE SELLS THE GALLIC PORTFOLIO, CONSISTING OF FIVE ASSETS IN THE LYON REGION
🤖 Analyzing with LLM...
🤖 Streaming LLM response....................................................................................................... Done!
✅ LLM analysis completed successfully
✅ Article 2 processed successfully

📰 Processing article 3/30...
📄 Title: KKR acquires $7.2 Billion Portfolio of Prime Recreational Vehicle Loans
🤖 Analyzing with LLM...
🤖 Streaming LLM response............................................................................................